# FT-900 留出期原件行与月份身份审核

仅打开 44 份已验证字节；年终 December and Annual 标题按事前规则接受，不读取市场结果。

In [ ]:
from pathlib import Path
import hashlib,json,re,datetime,fitz
root=Path.cwd();card=root/'research/census_ft900_semiconductor_export_source_20261023';plan=json.loads((card/'held_row_identity_plan.json').read_text());parent=json.loads((card/plan['parent_source_result']).read_text());readback=json.loads((card/plan['parent_readback']).read_text());manifest=json.loads((card/'held_url_manifest.json').read_text())
assert parent['target_months']==readback['PDF_hashes_verified']==len(manifest['month_urls'])==44
ledger=[json.loads(x) for x in (card/plan['source_receipts']).read_text().splitlines()];finished={x['month']:x for x in ledger if x['event'] in ('completed','reused')};assert len(finished)==44
rows=[]
for month in sorted(finished):
    receipt=finished[month];path=root/receipt['local_path'];payload=path.read_bytes();assert len(payload)==receipt['bytes'] and hashlib.sha256(payload).hexdigest()==receipt['sha256']
    doc=fitz.open(stream=payload,filetype='pdf');first=' '.join(doc[0].get_text().split());expected=datetime.datetime.strptime(month,'%Y-%m').strftime('%B %Y')
    title=re.search(r'U\.S\. INTERNATIONAL TRADE IN GOODS AND SERVICES,?\s+([A-Za-z]+)(\s+AND ANNUAL)?\s+(\d{4})',first,re.I)
    printed=(title.group(1)+' '+title.group(3)) if title else None;year_end=bool(title and title.group(2));identity=bool(printed and printed.lower()==expected.lower() and (not year_end or month.endswith('-12')))
    time=bool(re.search(r'8:30\s*(?:A\.?M\.?|AM)',first,re.I))
    candidates=[]
    for page_no,page in enumerate(doc,1):
        content=page.get_text();normalized=' '.join(content.split())
        if 'Exhibit 7.' not in normalized or 'Exports of Goods by End-Use Category and Commodity' not in normalized:continue
        lines=[x.strip() for x in content.splitlines()]
        for idx,line in enumerate(lines):
            if line=='Semiconductors':candidates.append({'page':page_no,'tokens_after_label':lines[idx+1:idx+7]})
    row={'reference_month':month,'source_pdf_sha256':receipt['sha256'],'source_bytes':len(payload),'source_url':receipt['url'],'printed_report_title_month':printed,'year_end_title':year_end,'month_identity_matches':identity,'first_page_0830_release_indicated':time,'row_candidate_count':len(candidates),'row_candidates':candidates,'status':'needs_review'}
    if len(candidates)==1:
        tokens=candidates[0]['tokens_after_label']
        if len(tokens)>=3 and all(re.fullmatch(r'-?[\d,]+',x) for x in tokens[:3]):
            v=[int(x.replace(',','')) for x in tokens[:3]];row.update(status='unique_numeric_Exhibit7_row',page=candidates[0]['page'],current_month_million_usd=v[0],prior_month_same_release_million_usd=v[1],printed_change_million_usd=v[2],printed_change_minus_rounded_arithmetic=v[2]-(v[0]-v[1]))
    rows.append(row)
exceptions=[x['reference_month'] for x in rows if x['status']!='unique_numeric_Exhibit7_row' or not x['month_identity_matches'] or not x['first_page_0830_release_indicated']]
result={'status':'all_44_held_rows_and_report_identities_verified' if not exceptions else 'held_row_or_identity_exceptions','months':44,'unique_numeric_Exhibit7_rows':sum(x['status']=='unique_numeric_Exhibit7_row' for x in rows),'matching_report_months':sum(x['month_identity_matches'] for x in rows),'first_page_0830_headers':sum(x['first_page_0830_release_indicated'] for x in rows),'year_end_titles':sum(x['year_end_title'] for x in rows),'exceptions':exceptions,'historical_first_public_bytes_qualified':False,'Qlib_factor_rows':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'rows':rows}
(card/'held_row_identity_audit.json').write_text(json.dumps(result,indent=2)+'\n');print({k:result[k] for k in ('status','unique_numeric_Exhibit7_rows','matching_report_months','first_page_0830_headers','year_end_titles','exceptions')})

## 结论边界

来源行和月份身份可用于后续可用性及信息门设计，不等于首次公开字节、独立前向日或 SOXX 超额。